In [61]:
from yaml import warnings
import warnings 
warnings.filterwarnings('ignore')
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('GK_Questions.pdf')
pages = loader.load()


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
spliter = RecursiveCharacterTextSplitter(chunk_size=500 , chunk_overlap = 160)


lines = spliter.split_documents(pages)
chunk = [i.page_content for i in lines]
metadata = [i.metadata for i in lines]


In [ ]:
import chromadb 
from chromadb.utils import embedding_functions 

embedding = embedding_functions.SentenceTransformerEmbeddingFunction('all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path='./Reranker_with_RAG')
collection = client.get_or_create_collection(name="GK_QA_clean", embedding_function=embedding)

collection.add(
    documents=chunk,
    ids=[str(i) for i in range(len(chunk))],
    metadatas=[{"source": "gk.pdf"} for _ in chunk]
)

collection.count()

from rank_bm25 import BM25Okapi 
tokens = [c.split() for c in chunk]
bm250 = BM25Okapi(tokens)

from sentence_transformers import CrossEncoder 
reranker  = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


In [ ]:
def hybrid_retrive(query:str , k:int=10):
    result = collection.query(query_texts=[query],n_results=20)
    document = result['documents'][0]
    distance = result['distances'][0]

    thresold = 1.0 
    
    dense_corpus= []
    for doc , d in zip(document,distance):
        if thresold > d :
            dense_corpus.append(doc)

    score = bm250.get_scores(query.split())

    def get_top_tokens(score , k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index , key= lambda x:x[1],reverse=True)
        return [ n for n , s in index_sorted[:k]]
    top_token = get_top_tokens(score=score , k=10)

    clone_top_token = [chunk[i] for i in top_token]

    rrn_token = {}
    for rank , doc in enumerate(dense_corpus):
        rrn_token[doc]=rrn_token.get(doc,0)+1/(60+rank)
    for rank , doc in enumerate(clone_top_token):
        rrn_token[doc]=rrn_token.get(doc,0)+1/(60+rank)

    merge = sorted(rrn_token.items(),key=lambda x : x[1] , reverse=True)
    top_docs = [doc for doc , _ in merge[:5]]
    

    if not top_docs:
        return []
    pairs =[[query,doc] for doc in top_docs]
    scores = reranker.predict(pairs)
    rerank = [doc for _ , doc in sorted(zip(scores,top_docs),key=lambda x : x[0],reverse=True)]

    return rerank[:5]


In [77]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()
key = os.getenv("GROQ_API_KEY")

chat_model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    api_key=key,
)

questions = [
    "What is the largest country",
    "What is the full form of the EU's parliament?",
]

for question in questions:
    answers = hybrid_retrive(query=question)

    prompt = f"""
Use the context to answer the question, even if the wording differs slightly.
Return ONLY the final answer.
Do not include labels like "Ans:".
If the context does not contain enough information, say "Not found in the provided documents."

Context:
{answers}

Question:
{question}
"""


    response = chat_model.invoke(prompt)

    print(f"Q: {question}")
    print(f"A: {response.content}\n")


Q: What is the largest country
A: Russia

Q: What is the full form of the EU's parliament?
A: European Parliament

